In [1]:
import requests
import pandas as pd
import time
import re


def classificar_empresa(nome):
    """
    Classifica empresa como SPE ou Incorporadora provável
    """
    nome = (nome or "").upper()

    padroes_spe = [
        "SPE",
        "EMPREENDIMENTO",
        "IMOBILIARIO",
        "RESIDENCIAL",
        "LTDA SCP"
    ]

    if any(p in nome for p in padroes_spe):
        return "SPE_PROVAVEL"

    return "INCORPORADORA_PROVAVEL"


def buscar_incorporadoras_ativas_dataframe(uf=None, limite_total=500):
    base_url = "https://minhareceita.org/"
    cnae = "4110700"

    resultados = []
    offset = 0
    limite_pagina = 100  # pode aumentar (até 1000 dependendo da API)

    print(f"Buscando dados (UF={uf}, limite_total={limite_total})...")

    while len(resultados) < limite_total:
        params = {
            "cnae": cnae,
            "limit": limite_pagina,
            "offset": offset
        }

        if uf:
            params["uf"] = uf

        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            empresas = data.get("data", [])
            if not empresas:
                break

            for emp in empresas:
                if emp.get("situacao_cadastral") == 2:
                    emp["tipo_empresa"] = classificar_empresa(
                        emp.get("razao_social") or emp.get("nome_fantasia")
                    )
                    resultados.append(emp)

            print(f"Coletados: {len(resultados)}")

            offset += limite_pagina
            time.sleep(0.5)  # evitar rate limit

        except Exception as e:
            print(f"Erro: {e}")
            break

    if not resultados:
        return pd.DataFrame()

    df = pd.DataFrame(resultados)

    # 🔥 Limpeza e enriquecimento
    colunas = [
        'cnpj', 'razao_social', 'nome_fantasia',
        'municipio', 'uf',
        'cnae_fiscal_descricao',
        'data_inicio_atividade',
        'tipo_empresa'
    ]

    df = df[colunas].copy()

    df.rename(columns={
        'cnpj': 'CNPJ',
        'razao_social': 'Razão Social',
        'nome_fantasia': 'Nome Fantasia',
        'municipio': 'Município',
        'uf': 'UF',
        'cnae_fiscal_descricao': 'CNAE',
        'data_inicio_atividade': 'Início',
        'tipo_empresa': 'Classificação'
    }, inplace=True)

    return df

In [2]:
df = buscar_incorporadoras_ativas_dataframe(uf="PR", limite_total=500)

# Separar grupos
df_spe = df[df["Classificação"] == "SPE_PROVAVEL"]
df_real = df[df["Classificação"] == "INCORPORADORA_PROVAVEL"]

print("SPE:", len(df_spe))
print("Incorporadoras reais:", len(df_real))

Buscando dados (UF=PR, limite_total=500)...
Coletados: 69
Coletados: 138
Coletados: 207
Coletados: 276
Coletados: 345
Coletados: 414
Coletados: 483
Coletados: 552
SPE: 160
Incorporadoras reais: 392


In [5]:
# 1. Função para buscar QSA

In [6]:
import requests
import time

def buscar_qsa(cnpj):
    url = f"https://minhareceita.org/{cnpj}"
    
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        socios = data.get("qsa", [])
        resultado = []

        for s in socios:
            resultado.append({
                "cnpj": cnpj,
                "socio_nome": s.get("nome_socio"),
                "qualificacao": s.get("qualificacao_socio")
            })

        return resultado

    except Exception as e:
        print(f"Erro ao buscar QSA {cnpj}: {e}")
        return []

In [7]:
# 2. Montar base de sócios

In [8]:
def montar_base_socios(df_empresas):
    socios_total = []

    for i, row in df_empresas.iterrows():
        cnpj = row["CNPJ"]

        socios = buscar_qsa(cnpj)
        socios_total.extend(socios)

        if i % 10 == 0:
            print(f"Processados {i} CNPJs")

        time.sleep(0.5)

    return socios_total

In [9]:
# 3. Criar grafo (agrupamento)

In [10]:
import pandas as pd

def agrupar_por_socios(lista_socios):
    df = pd.DataFrame(lista_socios)

    # remove sócios vazios
    df = df[df["socio_nome"].notna()]

    # agrupar
    grupos = df.groupby("socio_nome")["cnpj"].apply(list).reset_index()

    # filtrar sócios com mais de 1 empresa
    grupos["qtd_empresas"] = grupos["cnpj"].apply(len)
    grupos = grupos[grupos["qtd_empresas"] > 1]

    return grupos.sort_values("qtd_empresas", ascending=False)

In [11]:
# 4. Uso completo

In [12]:
df_empresas = buscar_incorporadoras_ativas_dataframe(uf="PR", limite_total=200)

socios = montar_base_socios(df_empresas)

df_grupos = agrupar_por_socios(socios)


Buscando dados (UF=PR, limite_total=200)...
Coletados: 69
Coletados: 138
Coletados: 207
Processados 0 CNPJs
Processados 10 CNPJs
Processados 20 CNPJs
Processados 30 CNPJs
Processados 40 CNPJs
Processados 50 CNPJs
Processados 60 CNPJs
Processados 70 CNPJs
Processados 80 CNPJs
Processados 90 CNPJs
Processados 100 CNPJs
Processados 110 CNPJs
Processados 120 CNPJs
Processados 130 CNPJs
Processados 140 CNPJs
Processados 150 CNPJs
Processados 160 CNPJs
Processados 170 CNPJs
Processados 180 CNPJs
Processados 190 CNPJs
Processados 200 CNPJs


In [15]:
df_grupos

,socio_nome,cnpj,qtd_empresas
0,A YOSHII ENGENHARIA E CONSTRUCOES LTDA,"[10910748004091, 10910748004091, 10910748004091]",3
1,A. YOSHII PARTICIPACOES LTDA,"[10910748004091, 10910748004091, 10910748004091]",3
2,ADRIANA GUIMARAES DEMETERCO,"[00650202000189, 00650202000189, 00650202000189]",3
3,AJG PARTICIPACOES IMOBILIARIAS LTDA,"[42908452000116, 42908452000116, 42908452000116]",3
4,ALEXANDRE CIRINO DOS SANTOS,"[17079502000152, 17079502000152, 17079502000152]",3
...,...,...,...
163,VALMIR SOUZA,"[13762095000122, 13762095000122, 13762095000122]",3
164,VITOR BATISTELLA DE GODOES,"[41999763000175, 41999763000175, 41999763000175]",3
165,VOLMAR SANDER,"[30750259000110, 30750259000110, 30750259000110]",3
166,WANDER THIAGO RONCAGLIO,"[14978947000186, 14978947000186, 14978947000186]",3
